# Dr.JE -> Experiment2

In [2]:
import random
import shutil
from pathlib import Path


In [3]:
# Input (JE raw)
SRC = Path("FromDrJE")
SRC_IMG = SRC / "images"
SRC_LBL = SRC / "labels"

# Output (Experiment2)
OUT = Path("Experiment2")

TEST_IMG = OUT / "test_30/images"
TEST_LBL = OUT / "test_30/labels"

TRAIN_IMG = OUT / "train_70/train/images"
TRAIN_LBL = OUT / "train_70/train/labels"

VAL_IMG = OUT / "train_70/val/images"
VAL_LBL = OUT / "train_70/val/labels"

ALL_OUT = [TEST_IMG, TEST_LBL, TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL]

for d in ALL_OUT:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Folders ready")


✅ Folders ready


In [4]:
IMG_EXTS = {".jpg", ".jpeg", ".png"}

images = [p for p in SRC_IMG.iterdir() if p.suffix.lower() in IMG_EXTS]
images.sort()

pairs = []
missing_label_files = 0

for img in images:
    lbl = SRC_LBL / (img.stem + ".txt")
    if not lbl.exists():
        missing_label_files += 1
        pairs.append((img, None))  # negative (no label file)
    else:
        pairs.append((img, lbl))

print(f"Images found: {len(images)}")
print(f"Missing label files (will become negatives): {missing_label_files}")


Images found: 73
Missing label files (will become negatives): 0


In [5]:
random.seed(42)
random.shuffle(pairs)

n_total = len(pairs)
n_test = int(0.30 * n_total)

test_pairs = pairs[:n_test]
train70_pairs = pairs[n_test:]

split_idx = int(0.80 * len(train70_pairs))
train_pairs = train70_pairs[:split_idx]
val_pairs = train70_pairs[split_idx:]

print(f"Test 30%:  {len(test_pairs)}")
print(f"Train 70%: {len(train70_pairs)}")
print(f"  Train:   {len(train_pairs)}")
print(f"  Val:     {len(val_pairs)}")


Test 30%:  21
Train 70%: 52
  Train:   41
  Val:     11


In [6]:
def read_label_text(label_path: Path | None) -> str:
    if label_path is None:
        return ""  # negative
    txt = label_path.read_text()
    return txt if txt.endswith("\n") or txt == "" else (txt + "\n")

def write_split(pairs, img_out: Path, lbl_out: Path):
    pos = 0
    neg = 0

    for img, lbl in pairs:
        shutil.copy2(img, img_out / img.name)

        out_lbl_path = lbl_out / (img.stem + ".txt")
        txt = read_label_text(lbl)
        out_lbl_path.write_text(txt)

        if txt.strip():
            pos += 1
        else:
            neg += 1

    return pos, neg

pos_te, neg_te = write_split(test_pairs, TEST_IMG, TEST_LBL)
pos_tr, neg_tr = write_split(train_pairs, TRAIN_IMG, TRAIN_LBL)
pos_va, neg_va = write_split(val_pairs, VAL_IMG, VAL_LBL)

print(f"✅ test_30  positives: {pos_te}, negatives: {neg_te}")
print(f"✅ train    positives: {pos_tr}, negatives: {neg_tr}")
print(f"✅ val      positives: {pos_va}, negatives: {neg_va}")


✅ test_30  positives: 21, negatives: 0
✅ train    positives: 41, negatives: 0
✅ val      positives: 11, negatives: 0


In [7]:
def stems_in(folder: Path, exts):
    return {p.stem for p in folder.iterdir() if p.suffix.lower() in exts}

# Check image/label matching
for name, img_dir, lbl_dir in [
    ("test_30", TEST_IMG, TEST_LBL),
    ("train", TRAIN_IMG, TRAIN_LBL),
    ("val", VAL_IMG, VAL_LBL),
]:
    img_stems = stems_in(img_dir, {".jpg",".jpeg",".png"})
    lbl_stems = stems_in(lbl_dir, {".txt"})
    print(f"\n--- {name} ---")
    print("Images:", len(img_stems), "Labels:", len(lbl_stems))
    print("✅ missing labels:", len(img_stems - lbl_stems))
    print("✅ missing images:", len(lbl_stems - img_stems))

# Check split overlap
test_stems = stems_in(TEST_IMG, {".jpg",".jpeg",".png"})
train_stems = stems_in(TRAIN_IMG, {".jpg",".jpeg",".png"})
val_stems = stems_in(VAL_IMG, {".jpg",".jpeg",".png"})

print("\n--- overlap checks ---")
print("test ∩ train:", len(test_stems & train_stems))
print("test ∩ val:  ", len(test_stems & val_stems))
print("train ∩ val: ", len(train_stems & val_stems))



--- test_30 ---
Images: 21 Labels: 21
✅ missing labels: 0
✅ missing images: 0

--- train ---
Images: 41 Labels: 41
✅ missing labels: 0
✅ missing images: 0

--- val ---
Images: 11 Labels: 11
✅ missing labels: 0
✅ missing images: 0

--- overlap checks ---
test ∩ train: 0
test ∩ val:   0
train ∩ val:  0
